# Level 3 — 불균형 대응 및 고급 Augmentation

**목표**: 다수 클래스의 정확도를 크게 희생하지 않으면서, 소수 클래스 (foggy / snowy / dawn-dusk) 의 성능을 끌어올립니다.

다음 축에서 **최소 2가지 이상** 의 기법을 적용하세요.
- Loss-level: Weighted CE, Focal Loss, LDAM, Class-Balanced Loss
- Sampling-level: class-balanced sampler
- Augmentation-level: RandAugment, Mixup, CutMix

Level 1 / 2 에서 가장 좋았던 백본을 base 로 사용하세요. wandb 를 사용하면 여러 기법의 비교 Run 을 같은 프로젝트에 모아 볼 수 있어 편리합니다.

In [1]:
import os

repo_name = "2026-HYU-AUE8088-PA2"
repo_path = f"/content/{repo_name}"
repo_url = "https://github.com/Jieunn-Kim/2026-HYU-AUE8088-PA2.git"

if not os.path.exists(repo_path):
    !git clone {repo_url} {repo_path}

%cd {repo_path}

Cloning into '/content/2026-HYU-AUE8088-PA2'...
remote: Enumerating objects: 56, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (13/13), done.
remote: Total 56 (delta 6), reused 2 (delta 2), pack-reused 41 (from 2)
Receiving objects: 100% (56/56), 82.57 KiB | 13.76 MiB/s, done.
Resolving deltas: 100% (15/15), done.
/content/2026-HYU-AUE8088-PA2


In [2]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [3]:

import shutil

vit_backup = "/content/drive/MyDrive/AUE8088_PA2/source_backup/vit.py"
vit_target = f"{repo_path}/src/models/vit.py"

shutil.copy2(vit_backup, vit_target)
print("vit.py 복원 완료")

vit.py 복원 완료


In [4]:
%load_ext autoreload
%autoreload 2

!pip install -q -r requirements.txt

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.4/26.4 MB 96.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 134.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 97.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 76.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, per_class_prf, CLASS_NAMES
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.datasets.samplers import class_balanced_sampler
from src.losses.imbalanced import FocalLoss, ClassBalancedLoss, LDAMLoss, weighted_cross_entropy
from src.augment.mix import mixup_data, cutmix_data, mixed_loss
from src.models.vit import vit_small_patch16_224

SEED = 42
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [6]:
import wandb; wandb.login()   # API key 입력

WANDB_PROJECT = "aue8088-pa2"   # 비활성화하려면 None
WANDB_TAGS    = ["level3"]

EXPERIMENT_NAME = "focal+sampler"

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: jieunnkim (jieunnkim-hanyang-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [7]:
DATA_ROOT = "../data/set_a"
BATCH = 64

# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
# ../data/set_a 가 없으면 zip 을 받아 상위 폴더에 압축 해제 → ../data/set_a, ../data/set_b 생성.
import os, sys, zipfile, subprocess

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."   # zip 내부 최상위가 data/ 이므로 상위 폴더에 풀면 ../data/... 가 됨

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

train_ds = BDDAttrDataset(DATA_ROOT, "train", transform=train_transform())
val_ds   = BDDAttrDataset(DATA_ROOT, "val",   transform=eval_transform())

# 옵션 A — 가장 불균형이 심한 weather 속성 기준 class-balanced sampler 사용
# sampler = class_balanced_sampler(train_ds, attribute="weather")
# train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, num_workers=2, pin_memory=True)
# val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

# Focal Loss 단독 실험용
train_loader_plain = DataLoader(
    train_ds,
    batch_size=BATCH,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
)

# Weather 기준 class-balanced sampler 실험용
weather_sampler = class_balanced_sampler(
    train_ds,
    attribute="weather",
)

train_loader_weather = DataLoader(
    train_ds,
    batch_size=BATCH,
    sampler=weather_sampler,
    num_workers=2,
    pin_memory=True,
)

# 모든 실험에서 동일하게 사용
val_loader = DataLoader(
    val_ds,
    batch_size=BATCH,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

데이터셋 zip 다운로드 중...


Downloading...
From (original): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK
From (redirected): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK&confirm=t&uuid=8bbccbf5-6765-4e00-9921-132638718d66
To: /content/aue8088_pa2_data.zip
100%|██████████| 243M/243M [00:02<00:00, 82.0MB/s]


압축 해제 중...
완료 → ../data/set_a


In [8]:
# 옵션 B — 속성별로 다른 loss 적용. 가장 불균형이 심한 속성에 가장 강한 loss 사용.
samples_w = train_ds.class_counts("weather")
samples_s = train_ds.class_counts("scene")
samples_t = train_ds.class_counts("timeofday")

loss_fns = {
    "weather":   FocalLoss(gamma=2.0).to(device),
    "scene":     ClassBalancedLoss(samples_s).to(device),
    "timeofday": nn.CrossEntropyLoss(),
}

# 직접 구현한 ViT-S/16 생성 후 ImageNet pretrained 가중치 로드
PRETRAINED_SOURCE = "Facebook Research DeiT-S/16, ImageNet-1K"
PRETRAINED_DIR = "/content/drive/MyDrive/AUE8088_PA2/pretrained"

os.makedirs(PRETRAINED_DIR, exist_ok=True)

DEIT_URL = (
    "https://dl.fbaipublicfiles.com/deit/"
    "deit_small_patch16_224-cd65a155.pth"
)

checkpoint = torch.hub.load_state_dict_from_url(
    DEIT_URL,
    model_dir=PRETRAINED_DIR,
    map_location="cpu",
    check_hash=True,
)

pretrained_state = checkpoint["model"]

model = vit_small_patch16_224()
model_state = model.state_dict()
mapped_state = {}

for key, value in pretrained_state.items():
    new_key = key

    if new_key.startswith("module."):
        new_key = new_key[len("module."):]

    new_key = new_key.replace(".mlp.fc1.", ".mlp.0.")
    new_key = new_key.replace(".mlp.fc2.", ".mlp.3.")

    # ImageNet classification head는 사용하지 않음
    if new_key.startswith("head."):
        continue

    if (
        new_key in model_state
        and model_state[new_key].shape == value.shape
    ):
        mapped_state[new_key] = value

missing, unexpected = model.load_state_dict(
    mapped_state,
    strict=False,
)

print("Pretrained source:", PRETRAINED_SOURCE)
print("매칭된 pretrained tensor 수:", len(mapped_state))
print("Missing keys:", missing)
print("Unexpected keys:", unexpected)

model = model.to(device)

epochs = 25
optim = torch.optim.AdamW(
    model.parameters(),
    lr=5e-4,
    weight_decay=5e-2,
)

sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    optim,
    T_max=epochs,
)

logger = WandbLogger(
    project=WANDB_PROJECT,
    run_name=f"level3-{EXPERIMENT_NAME}",
    config={
        "backbone": "vit_s16_pretrained",
        "pretrained_source": PRETRAINED_SOURCE,
        "matched_pretrained_tensors": len(mapped_state),
        "sampler": "class_balanced(weather)",
        "loss": {
            "weather": "focal_g2.0",
            "scene": "cb_loss",
            "timeofday": "ce",
        },
        "epochs": epochs,
        "batch": BATCH,
        "lr": 5e-4,
        "weight_decay": 5e-2,
        "seed": SEED,
    },
    tags=WANDB_TAGS + [EXPERIMENT_NAME],
)

trainer = MultiTaskTrainer(
    model,
    optim,
    sched,
    loss_fns,
    device,
    TrainConfig(epochs=epochs),
    logger=logger,
)

Pretrained source: Facebook Research DeiT-S/16, ImageNet-1K
매칭된 pretrained tensor 수: 150
Missing keys: ['head.weather.weight', 'head.weather.bias', 'head.scene.weight', 'head.scene.bias', 'head.timeofday.weight', 'head.timeofday.bias']
Unexpected keys: []


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


In [9]:
# 옵션 A + C
# 옵션 A의 weather-balanced sampler와
# 옵션 B의 속성별 loss를 함께 사용하여 학습

history = trainer.fit(
    train_loader_weather,
    val_loader,
)

# 최종 validation 결과
val_pred, _, val_tgt, _ = collect_predictions(
    model,
    val_loader,
    device,
)

cms = confusion_matrices(val_pred, val_tgt)

for a in ATTRIBUTES:
    logger.log_confusion_matrix(
        f"final/cm_{a}",
        cms[a],
        CLASS_NAMES[a],
    )

logger.finish()

# Drive에 체크포인트 저장
save_dir = "/content/drive/MyDrive/AUE8088_PA2/checkpoints"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(
    save_dir,
    "level3_focal_cb_sampler_weather.pth",
)

torch.save(
    {
        "state_dict": model.state_dict(),
        "history": history,
        "backbone": "vit_s16_pretrained",
        "sampler": "class_balanced_weather",
        "loss": {
            "weather": "focal_gamma_2.0",
            "scene": "class_balanced_loss",
            "timeofday": "cross_entropy",
        },
    },
    save_path,
)

print("저장 완료:", save_path)

[epoch 01/25] train_loss=1.9422  val_avg_MF1=0.6064  per={'weather': 0.5038185153566256, 'scene': 0.5221799358976137, 'timeofday': 0.7932763256838596}


[epoch 02/25] train_loss=1.3047  val_avg_MF1=0.6095  per={'weather': 0.4797781672910965, 'scene': 0.558234332313884, 'timeofday': 0.7906341558514449}


[epoch 03/25] train_loss=1.0248  val_avg_MF1=0.6174  per={'weather': 0.5044525941346488, 'scene': 0.5824192182086919, 'timeofday': 0.7652193465166093}


[epoch 04/25] train_loss=0.9110  val_avg_MF1=0.6030  per={'weather': 0.4493918783921049, 'scene': 0.5677799009677184, 'timeofday': 0.7917441219956859}


[epoch 05/25] train_loss=0.8375  val_avg_MF1=0.6436  per={'weather': 0.5142107859868016, 'scene': 0.5969585072111743, 'timeofday': 0.8196392213211845}


...

[epoch 23/25] train_loss=0.0532  val_avg_MF1=0.6544  per={'weather': 0.5165661966272929, 'scene': 0.6245551264576708, 'timeofday': 0.8219663933949648}


[epoch 24/25] train_loss=0.0474  val_avg_MF1=0.6523  per={'weather': 0.5205685165709396, 'scene': 0.6293886230728335, 'timeofday': 0.8069806889682666}


[epoch 25/25] train_loss=0.0406  val_avg_MF1=0.6551  per={'weather': 0.5290177508147255, 'scene': 0.6293125704890411, 'timeofday': 0.8069806889682666}


epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
lr,████▇▇▇▆▆▆▅▅▄▄▃▃▃▂▂▂▁▁▁▁▁
train/loss,█▆▅▄▄▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val/avg_macro_f1,▁▂▃▁▆▇▃▇▄▅▄█▆█▅▆▆▇▇▇▆▇▇▇▇
val/mf1_scene,▁▃▄▃▄▇▄▆▆▄▄▇▅▇▆▇▅█▇█▆▆▆▆▆
val/mf1_timeofday,▄▄▁▄▆▆▁▆▅▇▅▆▆▃▃▃█▄▆▃▄▅▆▅▅
val/mf1_weather,▅▃▅▁▆▅▆▆▂▅▄▇▅█▅▅▅▆▅▆▇▇▆▆▇
epoch,25
lr,0
train/loss,0.04063
val/avg_macro_f1,0.6551


저장 완료: /content/drive/MyDrive/AUE8088_PA2/checkpoints/level3_focal_cb_sampler_weather.pth


In [10]:
# loss only
import gc
import os

# 이전 실험 메모리 정리
for name in ["model", "trainer", "optim", "sched", "loss_fns"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

# 동일한 seed와 동일한 pretrained 초기값 사용
set_seed(SEED, deterministic=True)

model = vit_small_patch16_224()
model.load_state_dict(mapped_state, strict=False)
model = model.to(device)

# Loss-level 기법만 적용
loss_fns = {
    "weather": FocalLoss(gamma=2.0).to(device),
    "scene": ClassBalancedLoss(samples_s).to(device),
    "timeofday": nn.CrossEntropyLoss(),
}

epochs = 25

optim = torch.optim.AdamW(
    model.parameters(),
    lr=5e-4,
    weight_decay=5e-2,
)

sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    optim,
    T_max=epochs,
)

logger = WandbLogger(
    project=WANDB_PROJECT,
    run_name="level3-loss-only",
    config={
        "backbone": "vit_s16_pretrained",
        "pretrained_source": PRETRAINED_SOURCE,
        "sampler": "none",
        "loss": {
            "weather": "focal_g2.0",
            "scene": "class_balanced_loss",
            "timeofday": "cross_entropy",
        },
        "epochs": epochs,
        "batch": BATCH,
        "lr": 5e-4,
        "weight_decay": 5e-2,
        "seed": SEED,
    },
    tags=WANDB_TAGS + ["loss-only"],
)

trainer = MultiTaskTrainer(
    model,
    optim,
    sched,
    loss_fns,
    device,
    TrainConfig(epochs=epochs),
    logger=logger,
)

# 일반 shuffle loader 사용: sampler 적용 안 함
history_loss_only = trainer.fit(
    train_loader_plain,
    val_loader,
)

val_pred, _, val_tgt, _ = collect_predictions(
    model,
    val_loader,
    device,
)

cms = confusion_matrices(val_pred, val_tgt)

for a in ATTRIBUTES:
    logger.log_confusion_matrix(
        f"final/cm_{a}",
        cms[a],
        CLASS_NAMES[a],
    )

logger.finish()

save_dir = "/content/drive/MyDrive/AUE8088_PA2/checkpoints"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(
    save_dir,
    "level3_loss_only.pth",
)

torch.save(
    {
        "state_dict": model.state_dict(),
        "history": history_loss_only,
        "backbone": "vit_s16_pretrained",
        "sampler": "none",
        "loss": {
            "weather": "focal_gamma_2.0",
            "scene": "class_balanced_loss",
            "timeofday": "cross_entropy",
        },
    },
    save_path,
)

print("저장 완료:", save_path)

[epoch 01/25] train_loss=1.8655  val_avg_MF1=0.5406  per={'weather': 0.2543580438405892, 'scene': 0.5677379710476299, 'timeofday': 0.7997033714014846}


[epoch 02/25] train_loss=1.4869  val_avg_MF1=0.6280  per={'weather': 0.429696866099922, 'scene': 0.6202750710206612, 'timeofday': 0.8339720114775758}


[epoch 03/25] train_loss=1.3677  val_avg_MF1=0.5878  per={'weather': 0.4114132112599153, 'scene': 0.5220559122998147, 'timeofday': 0.8300579583089811}


[epoch 04/25] train_loss=1.2742  val_avg_MF1=0.6310  per={'weather': 0.5128033747325078, 'scene': 0.5706788736301845, 'timeofday': 0.8094055672174849}


[epoch 05/25] train_loss=1.2105  val_avg_MF1=0.6629  per={'weather': 0.49290895112583666, 'scene': 0.6620443875373453, 'timeofday': 0.8337510585796885}


...

[epoch 23/25] train_loss=0.0351  val_avg_MF1=0.6572  per={'weather': 0.5370517039859128, 'scene': 0.6523941198975497, 'timeofday': 0.7820374618063304}


[epoch 24/25] train_loss=0.0321  val_avg_MF1=0.6534  per={'weather': 0.530364468589786, 'scene': 0.6554962284939402, 'timeofday': 0.7743970000643512}


[epoch 25/25] train_loss=0.0303  val_avg_MF1=0.6573  per={'weather': 0.5376326222564922, 'scene': 0.6598768997101726, 'timeofday': 0.7743970000643512}


epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
lr,████▇▇▇▆▆▆▅▅▄▄▃▃▃▂▂▂▁▁▁▁▁
train/loss,█▇▆▆▆▅▅▅▄▄▄▃▃▂▂▂▂▁▁▁▁▁▁▁▁
val/avg_macro_f1,▁▆▄▆█▇▆▆▆█▆███▇▇████▇██▇█
val/mf1_scene,▃▆▁▃█▄▆▅▄█▆▆▆▆▆▇▇▆▇█▆█▇██
val/mf1_timeofday,▄▇▇▅▇█▁▅▅▄▄▆▅▆▁▃▃▃▂▂▂▃▃▂▂
val/mf1_weather,▁▅▅▇▇▇▇▆▇▇▆▇▇█▇▇███▇█▇█▇█
epoch,25
lr,0
train/loss,0.03028
val/avg_macro_f1,0.6573


저장 완료: /content/drive/MyDrive/AUE8088_PA2/checkpoints/level3_loss_only.pth


In [11]:
#Sampler-only
# 이전 실험 메모리 정리
for name in ["model", "trainer", "optim", "sched", "loss_fns"]:
    if name in globals():
        del globals()[name]

gc.collect()
torch.cuda.empty_cache()

# 동일한 seed와 동일한 pretrained 초기값 사용
set_seed(SEED, deterministic=True)

model = vit_small_patch16_224()
model.load_state_dict(mapped_state, strict=False)
model = model.to(device)

# 기본 CE만 사용: 특수 loss 적용 안 함
loss_fns = {
    "weather": nn.CrossEntropyLoss(),
    "scene": nn.CrossEntropyLoss(),
    "timeofday": nn.CrossEntropyLoss(),
}

epochs = 25

optim = torch.optim.AdamW(
    model.parameters(),
    lr=5e-4,
    weight_decay=5e-2,
)

sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    optim,
    T_max=epochs,
)

logger = WandbLogger(
    project=WANDB_PROJECT,
    run_name="level3-sampler-only",
    config={
        "backbone": "vit_s16_pretrained",
        "pretrained_source": PRETRAINED_SOURCE,
        "sampler": "class_balanced(weather)",
        "loss": {
            "weather": "cross_entropy",
            "scene": "cross_entropy",
            "timeofday": "cross_entropy",
        },
        "epochs": epochs,
        "batch": BATCH,
        "lr": 5e-4,
        "weight_decay": 5e-2,
        "seed": SEED,
    },
    tags=WANDB_TAGS + ["sampler-only"],
)

trainer = MultiTaskTrainer(
    model,
    optim,
    sched,
    loss_fns,
    device,
    TrainConfig(epochs=epochs),
    logger=logger,
)

# Weather-balanced sampler loader 사용
history_sampler_only = trainer.fit(
    train_loader_weather,
    val_loader,
)

val_pred, _, val_tgt, _ = collect_predictions(
    model,
    val_loader,
    device,
)

cms = confusion_matrices(val_pred, val_tgt)

for a in ATTRIBUTES:
    logger.log_confusion_matrix(
        f"final/cm_{a}",
        cms[a],
        CLASS_NAMES[a],
    )

logger.finish()

save_dir = "/content/drive/MyDrive/AUE8088_PA2/checkpoints"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(
    save_dir,
    "level3_sampler_only.pth",
)

torch.save(
    {
        "state_dict": model.state_dict(),
        "history": history_sampler_only,
        "backbone": "vit_s16_pretrained",
        "sampler": "class_balanced_weather",
        "loss": {
            "weather": "cross_entropy",
            "scene": "cross_entropy",
            "timeofday": "cross_entropy",
        },
    },
    save_path,
)

print("저장 완료:", save_path)

[epoch 01/25] train_loss=2.2083  val_avg_MF1=0.6019  per={'weather': 0.4384838124973309, 'scene': 0.5599226022673092, 'timeofday': 0.8073138922591108}


[epoch 02/25] train_loss=1.4998  val_avg_MF1=0.6349  per={'weather': 0.5166975900231857, 'scene': 0.5813851572292377, 'timeofday': 0.806740225826621}


[epoch 03/25] train_loss=1.1901  val_avg_MF1=0.6446  per={'weather': 0.5208066885546485, 'scene': 0.5920790800026402, 'timeofday': 0.8208780958104825}


[epoch 04/25] train_loss=1.0231  val_avg_MF1=0.6565  per={'weather': 0.5138700177140624, 'scene': 0.627715189192009, 'timeofday': 0.8278412134913485}


[epoch 05/25] train_loss=0.9126  val_avg_MF1=0.6310  per={'weather': 0.5314887771784323, 'scene': 0.5913866364713317, 'timeofday': 0.7702126237058935}


...

[epoch 23/25] train_loss=0.0449  val_avg_MF1=0.6506  per={'weather': 0.5583942621199992, 'scene': 0.6261473942413862, 'timeofday': 0.7672684956238176}


[epoch 24/25] train_loss=0.0505  val_avg_MF1=0.6512  per={'weather': 0.5521056408340158, 'scene': 0.6464795230724824, 'timeofday': 0.7549062652416346}


[epoch 25/25] train_loss=0.0403  val_avg_MF1=0.6525  per={'weather': 0.5552083539433383, 'scene': 0.6473256194975597, 'timeofday': 0.7549062652416346}


epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
lr,████▇▇▇▆▆▆▅▅▄▄▃▃▃▂▂▂▁▁▁▁▁
train/loss,█▆▅▄▄▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁
val/avg_macro_f1,▁▄▅▆▄▅▃▅█▄▇▄▅▄▄▆▅▆▅▆▅▆▅▅▅
val/mf1_scene,▂▃▄▆▄▄▁▄▆▄▅▃▆▃▄█▆▇▇▆▅▇▅▇▇
val/mf1_timeofday,▅▅▆▆▂▄▇▅█▃▇▄▄▃▃▄▁▂▁▂▂▃▂▁▁
val/mf1_weather,▁▆▆▅▆▇▅▆█▇█▆▆▇▇▅▆█▇█▇████
epoch,25
lr,0
train/loss,0.04027
val/avg_macro_f1,0.65248


저장 완료: /content/drive/MyDrive/AUE8088_PA2/checkpoints/level3_sampler_only.pth


In [9]:
# 옵션 C

import torch
from torch import nn
from tqdm import tqdm


def step_with_mix(images, targets):
    """각 batch에 50% 확률로 Mixup 또는 CutMix 적용."""

    if torch.rand(1).item() < 0.5:
        mixed_images, targets_a, targets_b, lam = mixup_data(
            images,
            targets,
            alpha=0.2,
        )
        aug_name = "mixup"

    else:
        mixed_images, targets_a, targets_b, lam = cutmix_data(
            images,
            targets,
            alpha=1.0,
        )
        aug_name = "cutmix"

    logits = trainer.model(mixed_images)

    loss = mixed_loss(
        trainer.loss_fns,
        logits,
        targets_a,
        targets_b,
        lam,
        weights=trainer.cfg.loss_weights,
    )

    return loss, aug_name, lam


history = {
    "train_loss": [],
    "val_avg_mf1": [],
    "val_per_mf1": [],
}


for epoch in range(trainer.cfg.epochs):
    trainer.model.train()

    running_loss = 0.0
    n_batches = 0

    pbar = tqdm(
        train_loader_weather,
        desc=f"train mixcut e{epoch + 1}",
        leave=False,
    )

    for batch in pbar:
        images = batch["image"].to(
            trainer.device,
            non_blocking=True,
        )

        targets = {
            attribute: batch[attribute].to(
                trainer.device,
                non_blocking=True,
            )
            for attribute in ATTRIBUTES
        }

        trainer.optimizer.zero_grad(
            set_to_none=True
        )

        amp_enabled = (
            trainer.cfg.amp
            and trainer.device.type == "cuda"
        )

        with torch.amp.autocast(
            device_type=trainer.device.type,
            enabled=amp_enabled,
        ):
            loss, aug_name, lam = step_with_mix(
                images,
                targets,
            )

        trainer.scaler.scale(loss).backward()

        if trainer.cfg.grad_clip is not None:
            trainer.scaler.unscale_(
                trainer.optimizer
            )

            nn.utils.clip_grad_norm_(
                trainer.model.parameters(),
                trainer.cfg.grad_clip,
            )

        trainer.scaler.step(
            trainer.optimizer
        )
        trainer.scaler.update()

        running_loss += loss.item()
        n_batches += 1

        pbar.set_postfix({
            "loss": f"{loss.item():.4f}",
            "aug": aug_name,
            "lam": f"{lam:.3f}",
        })

    train_loss = (
        running_loss / max(n_batches, 1)
    )

    # augmentation 없는 validation set 평가
    val_metrics = trainer.evaluate(
        val_loader
    )

    history["train_loss"].append(
        train_loss
    )

    history["val_avg_mf1"].append(
        val_metrics["avg_macro_f1"]
    )

    history["val_per_mf1"].append(
        val_metrics["per_macro_f1"]
    )

    if trainer.scheduler is not None:
        trainer.scheduler.step()

    print(
        f"[epoch {epoch + 1:02d}/"
        f"{trainer.cfg.epochs}] "
        f"train_loss={train_loss:.4f} "
        f"val_avg_MF1="
        f"{val_metrics['avg_macro_f1']:.4f} "
        f"per={val_metrics['per_macro_f1']}"
    )

    log_payload = {
        "epoch": epoch + 1,
        "train/loss": train_loss,
        "val/avg_macro_f1":
            val_metrics["avg_macro_f1"],
        "lr":
            trainer.optimizer.param_groups[0]["lr"],
    }

    for attribute, value in (
        val_metrics["per_macro_f1"].items()
    ):
        log_payload[
            f"val/mf1_{attribute}"
        ] = value

    trainer.logger.log(
        log_payload,
        step=epoch,
    )

[epoch 01/25] train_loss=2.4710 val_avg_MF1=0.5577 per={'weather': 0.4073533982935384, 'scene': 0.48196696490594654, 'timeofday': 0.7837585441552534}


[epoch 02/25] train_loss=2.1011 val_avg_MF1=0.5378 per={'weather': 0.471739538390767, 'scene': 0.501952608794714, 'timeofday': 0.6395611814345991}


[epoch 03/25] train_loss=2.0517 val_avg_MF1=0.6349 per={'weather': 0.5106297813924933, 'scene': 0.5748933654628695, 'timeofday': 0.8192531546004834}


[epoch 04/25] train_loss=1.9019 val_avg_MF1=0.6121 per={'weather': 0.49639070704060023, 'scene': 0.545104905449274, 'timeofday': 0.7946978123841305}


[epoch 05/25] train_loss=1.7704 val_avg_MF1=0.6481 per={'weather': 0.5279227104150264, 'scene': 0.5578574062214977, 'timeofday': 0.8584237113648879}


...

[epoch 21/25] train_loss=0.9707 val_avg_MF1=0.6914 per={'weather': 0.5503793718568001, 'scene': 0.6822412023311912, 'timeofday': 0.8416390007299098}


[epoch 22/25] train_loss=1.0763 val_avg_MF1=0.6731 per={'weather': 0.5392976866131217, 'scene': 0.6547884409785419, 'timeofday': 0.8253165422976743}


[epoch 23/25] train_loss=1.0297 val_avg_MF1=0.6799 per={'weather': 0.5388559618625924, 'scene': 0.6685258775664681, 'timeofday': 0.8324209431263373}


[epoch 24/25] train_loss=0.9662 val_avg_MF1=0.6747 per={'weather': 0.5244260666061693, 'scene': 0.6716132975972163, 'timeofday': 0.8280361809773575}


[epoch 25/25] train_loss=0.9613 val_avg_MF1=0.6767 per={'weather': 0.5303174951695223, 'scene': 0.6716132975972163, 'timeofday': 0.8280361809773575}


In [17]:
# 저장
save_dir = (
    "/content/drive/MyDrive/"
    "AUE8088_PA2/checkpoints"
)

os.makedirs(
    save_dir,
    exist_ok=True,
)

save_path = os.path.join(
    save_dir,
    "level3_focal_cb_sampler_weather_mixcut.pth",
)

torch.save(
    {
        "state_dict":
            trainer.model.state_dict(),
        "history": history,
        "backbone":
            "vit_s16_pretrained",
        "sampler":
            "class_balanced_weather",
        "loss": {
            "weather":
                "focal_gamma_2.0",
            "scene":
                "class_balanced_loss",
            "timeofday":
                "cross_entropy",
        },
        "augmentation": {
            "mixup_alpha": 0.2,
            "cutmix_alpha": 1.0,
            "selection": "50:50",
        },
    },
    save_path,
)

print("저장 완료:", save_path)

trainer.logger.finish()

저장 완료: /content/drive/MyDrive/AUE8088_PA2/checkpoints/level3_focal_cb_sampler_weather_mixcut.pth


epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
lr,████▇▇▇▆▆▆▅▅▄▄▃▃▃▂▂▂▁▁▁▁▁
train/loss,█▇▆▅▅▅▅▄▄▄▄▃▄▃▂▃▂▂▂▂▁▁▂▂▁
val/avg_macro_f1,▁▄▄▅▅▅▅▇▆▆▅▇▆█▇▇▇▇▇▇█▇▇▇▇
val/mf1_scene,▃▂▁▄▅▄▅▆▆▆▆▇▅█▆▆▆▆▆▅▇▆▇▆▇
val/mf1_timeofday,▃▇▅▅▇▃▄▇▃▅▁▇▇█▆▅▇▇▆▇█████
val/mf1_weather,▁▄▇▆▅█▆▇█▇▇▆▆▇█▇▇█▇█▇▇▇▇▇
epoch,25
lr,0
train/loss,0.87638
val/avg_macro_f1,0.66799


In [10]:
# 학습 종료 후 — 속성별 confusion matrix + per-class F1 표를 wandb 에 업로드
val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
cms = confusion_matrices(val_pred, val_tgt)
prf = per_class_prf(val_pred, val_tgt)
for a in ATTRIBUTES:
    logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])
    rows = list(zip(prf[a]["class"], prf[a]["precision"], prf[a]["recall"], prf[a]["f1"], prf[a]["support"]))
    logger.log_table(f"final/prf_{a}", ["class", "P", "R", "F1", "support"], [list(r) for r in rows])
logger.finish()

epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
lr,████▇▇▇▆▆▆▅▅▄▄▃▃▃▂▂▂▁▁▁▁▁
train/loss,█▆▆▅▅▄▅▄▄▃▃▃▃▂▂▂▂▂▂▁▁▂▁▁▁
val/avg_macro_f1,▂▁▅▄▆▆▆▆▅▆▆▆▇▇▇▇▇▆▇██▇▇▇▇
val/mf1_scene,▁▂▄▃▄▄▇▆▄▆▄▆▆▇▇▆▆▇███▇███
val/mf1_timeofday,▆▁▇▆█▇▆▆▅▆▇▆▇▆▇▇▇▆▆▇▇▇▇▇▇
val/mf1_weather,▁▄▆▅▇▆▇▆▆▆▇▇▇█▇▇▇▇█▇█▇▇▇▇
epoch,25
lr,0
train/loss,0.96129
val/avg_macro_f1,0.67666


## 분석 (필수)

각 기법에 대해 **속성별 per-class F1 표** 를 작성하세요. 다음 항목을 강조해 주세요.
- 소수 클래스 (foggy / snowy / dawn-dusk) 의 적용 전후 성능 차이.
- 다수 클래스의 회귀 (regression) 발생 여부 — 그 trade-off 가 정당한지 논거.
- Sampling 과 Mixup / CutMix 의 상호작용 — 서로 도움이 되는지 충돌하는지.